In [60]:
import frust as ft
import pandas as pd
import numpy as np

from prism_pruner.graph_manipulations import graphize
from prism_pruner.pruner import prune_by_rmsd, prune_by_rmsd_rot_corr, prune_by_moment_of_inertia
from frust.schema import infer_group_columns

In [61]:
TS = "TS1"

In [62]:
components = ft.screen.read("../datasets/1m1c.csv")
systems = ft.screen.expand(components)
ts_guesses = ft.screen.create_ts_guesses(
    systems,
    ts_types=[TS],
    n_confs=50,
)

df = ts_guesses[TS].copy()
df_rpos = df[df["rpos"] == 4]

In [63]:
# ft.plot_conformers(df_rpos, mode="cluster", background_color="white")

In [69]:
def prism_prune_df(
    df,
    *,
    coords_col="coords_embedded",
    atoms_col="atoms",
    energy_col=None,
    group_cols=None,
    moi_max_deviation=1e-2,
    rmsd_max_rmsd=0.25,
    rmsd_max_dev=None,
):
    if group_cols is None:
        group_cols = infer_group_columns(df)

    kept = []

    for _, group in df.groupby(group_cols, dropna=False, sort=False):
        work = group.copy()

        if energy_col is not None:
            work = work.sort_values(energy_col, na_position="last")

        atoms = np.asarray(work.iloc[0][atoms_col], dtype=str)

        if not all(list(a) == list(atoms) for a in work[atoms_col]):
            raise ValueError("All rows in a pruning group must have the same atom order.")

        coords = np.stack([np.asarray(c, dtype=float) for c in work[coords_col]])

        active_coords = coords
        active_indices = np.arange(len(work))

        _, mask = prune_by_moment_of_inertia(
            active_coords,
            atoms,
            max_deviation=moi_max_deviation,
        )
        active_coords = active_coords[mask]
        active_indices = active_indices[mask]

        _, mask = prune_by_rmsd(
            active_coords,
            atoms,
            max_rmsd=rmsd_max_rmsd,
            max_dev=rmsd_max_dev,
        )
        active_coords = active_coords[mask]
        active_indices = active_indices[mask]

        graph = graphize(atoms, active_coords[0])
        _, mask = prune_by_rmsd_rot_corr(
            active_coords,
            atoms,
            graph,
            max_rmsd=rmsd_max_rmsd,
            max_dev=rmsd_max_dev,
        )
        active_indices = active_indices[mask]

        kept.append(work.iloc[active_indices])

    out = pd.concat(kept).sort_index()
    out.attrs.update(getattr(df, "attrs", {}))
    out.attrs["frust_prism_pruner"] = {
        "method": "prune_by_moment_of_inertia + prune_by_rmsd + prune_by_rmsd_rot_corr",
        "moi_max_deviation": moi_max_deviation,
        "rmsd_max_rmsd": rmsd_max_rmsd,
        "rmsd_max_dev": rmsd_max_dev,
        "input_rows": len(df),
        "output_rows": len(out),
        "dropped_rows": len(df) - len(out),
    }
    return out

In [70]:
pruned_df = prism_prune_df(df, rmsd_max_rmsd=1.5, rmsd_max_dev=None, moi_max_deviation=0.01)
c = ft.plot_conformers(pruned_df, mode="cluster", background_color="white", cluster_threshold=0, export_HTML="c.html", top_n=200, columns=2)
pruned_df.attrs["frust_prism_pruner"]

HTML export successful: c.html


{'method': 'prune_by_moment_of_inertia + prune_by_rmsd + prune_by_rmsd_rot_corr',
 'moi_max_deviation': 0.01,
 'rmsd_max_rmsd': 1.5,
 'rmsd_max_dev': None,
 'input_rows': 200,
 'output_rows': 26,
 'dropped_rows': 174}